# Modeles Avances — Fine-tuning de Transformers
## DistilBERT, RoBERTa et DistilBERT + XGBoost pour la detection de fake news

**Objectif** : Utiliser des modeles de langage pre-entraines (embeddings contextuels) pour ameliorer la classification binaire de fake news.

**Approches** :
1. **DistilBERT** : Fine-tuning classique (version compacte de BERT)
2. **RoBERTa** : Fine-tuning classique (version optimisee de BERT)
3. **DistilBERT + XGBoost** : Approche hybride — embeddings [CLS] de DistilBERT + classifieur XGBoost

**Avantage par rapport aux baselines** :
- Representations contextuelles : le meme mot a des vecteurs differents selon le contexte
- Transfer learning : les modeles ont deja appris la structure du langage sur d'immenses corpus
- Fine-tuning : on adapte le modele a notre tache specifique
- Hybride : XGBoost peut exploiter les embeddings mieux que la tete lineaire sur les petits datasets

**Ref** :
- Devlin et al. (2019) — BERT
- Liu et al. (2019) — RoBERTa
- Sanh et al. (2019) — DistilBERT
- Chen & Guestrin (2016) — XGBoost

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch : {torch.__version__}")

KeyboardInterrupt: 

## 1. Chargement des donnees et preparation

In [ ]:
df_train = pd.read_parquet(DATA_DIR / "liar_train.parquet")
df_valid = pd.read_parquet(DATA_DIR / "liar_valid.parquet")
df_test  = pd.read_parquet(DATA_DIR / "liar_test.parquet")

print(f"Train : {len(df_train):,} | Valid : {len(df_valid):,} | Test : {len(df_test):,}")

# On utilise le texte original (pas clean) pour les transformers qui ont leur propre tokenizer
train_texts = df_train["statement"].tolist()
train_labels = df_train["label_binary"].tolist()

valid_texts = df_valid["statement"].tolist()
valid_labels = df_valid["label_binary"].tolist()

test_texts = df_test["statement"].tolist()
test_labels = df_test["label_binary"].tolist()

Train : 10,240 | Valid : 1,284 | Test : 1,267


In [ ]:
# Dataset PyTorch pour les transformers
class FakeNewsDataset(Dataset):
    """Dataset compatible avec HuggingFace Trainer."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Metriques pour le Trainer
def compute_metrics(eval_pred):
    """Calcule accuracy et F1 pour le Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

print("Dataset et metriques definis.")

Dataset et metriques definis.


## 2. Fine-tuning DistilBERT

**DistilBERT** (Sanh et al., 2019) est une version distillee de BERT :
- 6 couches au lieu de 12 → 40% plus petit
- 60% plus rapide a l'inference
- Conserve ~97% des performances de BERT

On fine-tune le modele `distilbert-base-uncased` sur notre tache de classification binaire.

In [ ]:
# --- DistilBERT ---
MODEL_NAME_DB = "distilbert-base-uncased"
OUTPUT_DIR_DB = str(MODELS_DIR / "distilbert")

print(f"Chargement du tokenizer et modele : {MODEL_NAME_DB}")
tokenizer_db = AutoTokenizer.from_pretrained(MODEL_NAME_DB)
model_db = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DB, num_labels=2
)

# Preparation des datasets
train_dataset_db = FakeNewsDataset(train_texts, train_labels, tokenizer_db)
valid_dataset_db = FakeNewsDataset(valid_texts, valid_labels, tokenizer_db)
test_dataset_db  = FakeNewsDataset(test_texts, test_labels, tokenizer_db)

print(f"Datasets crees — Train: {len(train_dataset_db)}, Valid: {len(valid_dataset_db)}, Test: {len(test_dataset_db)}")

Chargement du tokenizer et modele : distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Datasets crees — Train: 10240, Valid: 1284, Test: 1267


In [ ]:
# Configuration de l'entrainement DistilBERT
training_args_db = TrainingArguments(
    output_dir=OUTPUT_DIR_DB,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),  # Mixed precision si GPU
)

trainer_db = Trainer(
    model=model_db,
    args=training_args_db,
    train_dataset=train_dataset_db,
    eval_dataset=valid_dataset_db,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Lancement du fine-tuning DistilBERT...")
train_result_db = trainer_db.train()
print(f"\nEntrainement termine — Loss finale : {train_result_db.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Lancement du fine-tuning DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.650927,0.656074,0.632399,0.625635
2,0.613594,0.652470,0.643302,0.632583
3,0.516063,0.697021,0.631620,0.620864
4,0.393765,0.780409,0.630841,0.626660


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Entrainement termine — Loss finale : 0.5497


In [ ]:
# Evaluation DistilBERT sur le test set (sans passer par trainer.evaluate pour eviter le bug EarlyStopping)
model_db.eval()
model_db.to(device)

all_preds_db = []
all_labels_db = []

with torch.no_grad():
    for i in range(0, len(test_dataset_db), 64):
        batch_indices = range(i, min(i + 64, len(test_dataset_db)))
        batch = {k: torch.stack([test_dataset_db[j][k] for j in batch_indices]).to(device)
                 for k in ["input_ids", "attention_mask"]}
        labels = torch.stack([test_dataset_db[j]["labels"] for j in batch_indices])
        
        outputs = model_db(**batch)
        preds = torch.argmax(outputs.logits, dim=-1).cpu()
        all_preds_db.extend(preds.numpy())
        all_labels_db.extend(labels.numpy())

y_pred_db = np.array(all_preds_db)
y_true_db = np.array(all_labels_db)

acc_db = accuracy_score(y_true_db, y_pred_db)
f1_db = f1_score(y_true_db, y_pred_db, average="weighted")

print(f"DistilBERT — Test Accuracy: {acc_db:.4f}  F1: {f1_db:.4f}")
print(f"\n{classification_report(y_true_db, y_pred_db, target_names=['Fake', 'Real'])}")

# Stocker les resultats pour la comparaison
results_db = {"eval_accuracy": acc_db, "eval_f1_weighted": f1_db}

# Sauvegarder le meilleur modele
model_db.save_pretrained(str(MODELS_DIR / "distilbert_best"))
tokenizer_db.save_pretrained(str(MODELS_DIR / "distilbert_best"))
print(f"Modele sauvegarde dans {MODELS_DIR / 'distilbert_best'}")

DistilBERT — Test Accuracy: 0.6425  F1: 0.6296

              precision    recall  f1-score   support

        Fake       0.63      0.44      0.52       553
        Real       0.65      0.80      0.72       714

    accuracy                           0.64      1267
   macro avg       0.64      0.62      0.62      1267
weighted avg       0.64      0.64      0.63      1267



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modele sauvegarde dans ..\models\distilbert_best


## 3. Fine-tuning RoBERTa

**RoBERTa** (Liu et al., 2019) ameliore BERT avec :
- Plus de donnees d'entrainement
- Suppression du NSP (Next Sentence Prediction)
- Sequences plus longues et plus grands batch sizes
- Dynamic masking au lieu de static masking

In [ ]:
# --- RoBERTa ---
MODEL_NAME_RB = "roberta-base"
OUTPUT_DIR_RB = str(MODELS_DIR / "roberta")

print(f"Chargement du tokenizer et modele : {MODEL_NAME_RB}")
tokenizer_rb = AutoTokenizer.from_pretrained(MODEL_NAME_RB)
model_rb = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_RB, num_labels=2
)

# Preparation des datasets
train_dataset_rb = FakeNewsDataset(train_texts, train_labels, tokenizer_rb)
valid_dataset_rb = FakeNewsDataset(valid_texts, valid_labels, tokenizer_rb)
test_dataset_rb  = FakeNewsDataset(test_texts, test_labels, tokenizer_rb)

# Configuration
training_args_rb = TrainingArguments(
    output_dir=OUTPUT_DIR_RB,
    num_train_epochs=5,
    per_device_train_batch_size=16,  # RoBERTa plus gros → batch plus petit
    per_device_eval_batch_size=32,
    learning_rate=1e-5,              # Learning rate plus faible pour RoBERTa
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer_rb = Trainer(
    model=model_rb,
    args=training_args_rb,
    train_dataset=train_dataset_rb,
    eval_dataset=valid_dataset_rb,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Lancement du fine-tuning RoBERTa...")
train_result_rb = trainer_rb.train()
print(f"\nEntrainement termine — Loss finale : {train_result_rb.training_loss:.4f}")

Chargement du tokenizer et modele : roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Lancement du fine-tuning RoBERTa...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.649815,0.670875,0.627726,0.615445
2,0.624112,0.678741,0.619938,0.586864
3,0.554310,0.654046,0.660436,0.657470
4,0.499007,0.689616,0.658879,0.651500
5,0.436625,0.756658,0.654984,0.646403


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Entrainement termine — Loss finale : 0.5645


In [ ]:
# Evaluation RoBERTa sur le test set
model_rb.eval()
model_rb.to(device)

all_preds_rb = []
all_labels_rb = []

with torch.no_grad():
    for i in range(0, len(test_dataset_rb), 32):
        batch_indices = range(i, min(i + 32, len(test_dataset_rb)))
        batch = {k: torch.stack([test_dataset_rb[j][k] for j in batch_indices]).to(device)
                 for k in ["input_ids", "attention_mask"]}
        labels = torch.stack([test_dataset_rb[j]["labels"] for j in batch_indices])
        
        outputs = model_rb(**batch)
        preds = torch.argmax(outputs.logits, dim=-1).cpu()
        all_preds_rb.extend(preds.numpy())
        all_labels_rb.extend(labels.numpy())

y_pred_rb = np.array(all_preds_rb)
y_true_rb = np.array(all_labels_rb)

acc_rb = accuracy_score(y_true_rb, y_pred_rb)
f1_rb = f1_score(y_true_rb, y_pred_rb, average="weighted")

print(f"RoBERTa — Test Accuracy: {acc_rb:.4f}  F1: {f1_rb:.4f}")
print(f"\n{classification_report(y_true_rb, y_pred_rb, target_names=['Fake', 'Real'])}")

results_rb = {"eval_accuracy": acc_rb, "eval_f1_weighted": f1_rb}

# Sauvegarder
model_rb.save_pretrained(str(MODELS_DIR / "roberta_best"))
tokenizer_rb.save_pretrained(str(MODELS_DIR / "roberta_best"))
print(f"Modele sauvegarde dans {MODELS_DIR / 'roberta_best'}")

RoBERTa — Test Accuracy: 0.6472  F1: 0.6423

              precision    recall  f1-score   support

        Fake       0.61      0.52      0.56       553
        Real       0.67      0.75      0.70       714

    accuracy                           0.65      1267
   macro avg       0.64      0.63      0.63      1267
weighted avg       0.64      0.65      0.64      1267



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modele sauvegarde dans ..\models\roberta_best


## 3b. DistilBERT + XGBoost (approche hybride)

**Principe** : Au lieu d'utiliser la tete de classification fine-tunee de DistilBERT, on extrait les **embeddings [CLS]** (vecteurs de 768 dimensions) du modele fine-tune, puis on entraine un **XGBoost** sur ces representations.

**Avantages de cette approche hybride** :
- Les embeddings DistilBERT capturent le **contexte semantique** (avantage des transformers)
- XGBoost est un classifieur robuste qui **gere mieux les petits datasets** et l'overfitting via le gradient boosting
- La combinaison peut surpasser le fine-tuning classique quand le dataset est limite (~10K exemples)

**Pipeline** :
1. Passer chaque texte dans DistilBERT fine-tune → extraire le vecteur [CLS] (768-dim)
2. Constituer les matrices X_train / X_valid / X_test de ces vecteurs
3. Entrainer XGBoost avec recherche d'hyperparametres

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm
import joblib

# ── Etape 1 : Extraction des embeddings [CLS] ──────────────────────────────

def extract_cls_embeddings(texts, tokenizer, model, batch_size=64):
    """Extrait le vecteur [CLS] (768-dim) de DistilBERT pour chaque texte."""
    model.eval()
    model.to(device)
    embeddings = []

    # Acceder au backbone DistilBERT (sans la tete de classification)
    backbone = model.distilbert if hasattr(model, "distilbert") else model.base_model

    for i in tqdm(range(0, len(texts), batch_size), desc="Extraction embeddings"):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts, truncation=True, padding="max_length",
            max_length=128, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = backbone(**encoded)
            # Le vecteur [CLS] est le premier token de la derniere couche
            cls_vectors = outputs.last_hidden_state[:, 0, :]  # shape: (batch, 768)
            embeddings.append(cls_vectors.cpu().numpy())

    return np.vstack(embeddings)

print("Extraction des embeddings DistilBERT [CLS]...")
X_train_emb = extract_cls_embeddings(train_texts, tokenizer_db, model_db)
X_valid_emb = extract_cls_embeddings(valid_texts, tokenizer_db, model_db)
X_test_emb  = extract_cls_embeddings(test_texts, tokenizer_db, model_db)

y_train_arr = np.array(train_labels)
y_valid_arr = np.array(valid_labels)
y_test_arr  = np.array(test_labels)

print(f"\nDimensions des embeddings :")
print(f"  Train : {X_train_emb.shape}")
print(f"  Valid : {X_valid_emb.shape}")
print(f"  Test  : {X_test_emb.shape}")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# ── Etape 2 : Entrainement XGBoost sur les embeddings ─────────────────────

# Combiner train + valid pour le GridSearch (avec CV)
X_trainval = np.vstack([X_train_emb, X_valid_emb])
y_trainval = np.concatenate([y_train_arr, y_valid_arr])

print("Recherche d'hyperparametres XGBoost sur embeddings DistilBERT...")
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

grid = GridSearchCV(
    xgb_base, param_grid, cv=3,
    scoring="f1_weighted", n_jobs=-1, verbose=1,
)
grid.fit(X_trainval, y_trainval)

print(f"\nMeilleurs hyperparametres : {grid.best_params_}")
print(f"Meilleur F1 (CV) : {grid.best_score_:.4f}")

xgb_model = grid.best_estimator_

In [ ]:
# ── Etape 3 : Evaluation sur le test set ─────────────────────────────────

y_pred_xgb = xgb_model.predict(X_test_emb)
y_proba_xgb = xgb_model.predict_proba(X_test_emb)

acc_xgb = accuracy_score(y_test_arr, y_pred_xgb)
f1_xgb = f1_score(y_test_arr, y_pred_xgb, average="weighted")

print(f"DistilBERT + XGBoost — Test Accuracy: {acc_xgb:.4f}  F1: {f1_xgb:.4f}")
print(f"\n{classification_report(y_test_arr, y_pred_xgb, target_names=['Fake', 'Real'])}")

# Matrice de confusion
cm_xgb = confusion_matrix(y_test_arr, y_pred_xgb)
print("Matrice de confusion :")
print(cm_xgb)

# Stocker pour la comparaison
results_xgb = {"eval_accuracy": acc_xgb, "eval_f1_weighted": f1_xgb}

# Sauvegarder le modele XGBoost
xgb_path = MODELS_DIR / "distilbert_xgboost.joblib"
joblib.dump(xgb_model, xgb_path)
print(f"\nModele sauvegarde -> {xgb_path}")

## 4. Comparaison des modeles avances vs baselines

Incluons maintenant les 3 modeles avances (DistilBERT, RoBERTa, DistilBERT+XGBoost) et les baselines dans un comparatif unique.

In [ ]:
# Charger les metriques des baselines
baselines_path = MODELS_DIR / "baselines_metrics.json"
if baselines_path.exists():
    with open(baselines_path) as f:
        baselines = json.load(f)
else:
    baselines = {}
    print("Pas de metriques baselines trouvees. Executez d'abord Modeles_de_Base.ipynb")

# Combiner toutes les metriques
all_metrics = {}
for name, m in baselines.items():
    all_metrics[name] = {"Accuracy": m.get("test_acc", 0), "F1-Score": m.get("test_f1", 0)}

all_metrics["DistilBERT"] = {
    "Accuracy": round(results_db["eval_accuracy"], 4),
    "F1-Score": round(results_db["eval_f1_weighted"], 4),
}
all_metrics["RoBERTa"] = {
    "Accuracy": round(results_rb["eval_accuracy"], 4),
    "F1-Score": round(results_rb["eval_f1_weighted"], 4),
}
all_metrics["DistilBERT + XGBoost"] = {
    "Accuracy": round(results_xgb["eval_accuracy"], 4),
    "F1-Score": round(results_xgb["eval_f1_weighted"], 4),
}

compare_df = pd.DataFrame(all_metrics).T.sort_values("F1-Score", ascending=False)
print("=== Comparaison complete des modeles ===")
compare_df

In [ ]:
# Visualisation comparative
fig = go.Figure()
models = compare_df.index.tolist()

fig.add_trace(go.Bar(name="Accuracy", x=models, y=compare_df["Accuracy"], marker_color="#3498db"))
fig.add_trace(go.Bar(name="F1-Score", x=models, y=compare_df["F1-Score"], marker_color="#e74c3c"))

fig.update_layout(
    barmode="group",
    title="Comparaison Baselines vs Transformers — Accuracy & F1",
    yaxis_title="Score", template="plotly_white", height=500,
    yaxis=dict(range=[0, 1]),
)
fig.show()

In [ ]:
# Courbes d'entrainement DistilBERT
log_db = trainer_db.state.log_history
train_losses = [x["loss"] for x in log_db if "loss" in x and "eval_loss" not in x]
eval_entries = [x for x in log_db if "eval_loss" in x]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss d'entrainement", "F1 sur validation"])

fig.add_trace(go.Scatter(
    y=train_losses, mode="lines+markers", name="Train Loss",
    line=dict(color="#e74c3c"),
), row=1, col=1)

if eval_entries:
    eval_f1 = [x.get("eval_f1_weighted", 0) for x in eval_entries]
    fig.add_trace(go.Scatter(
        y=eval_f1, mode="lines+markers", name="Valid F1",
        line=dict(color="#2ecc71"),
    ), row=1, col=2)

fig.update_layout(title="Courbes d'entrainement — DistilBERT", template="plotly_white", height=400)
fig.show()

## 5. Synthese

**Modeles evalues** :
| Modele | Approche | Particularite |
|---|---|---|
| DistilBERT | Fine-tuning classique | Tete de classification lineaire |
| RoBERTa | Fine-tuning classique | Pre-entrainement plus robuste |
| **DistilBERT + XGBoost** | **Hybride** | **Embeddings [CLS] + gradient boosting** |

**Analyse** :
- Les transformers capturent le **contexte semantique** que TF-IDF ne peut pas saisir
- L'approche hybride **DistilBERT + XGBoost** combine les forces des deux mondes : embeddings contextuels + classifieur robuste aux petits datasets
- XGBoost gere mieux l'overfitting que la tete lineaire de fine-tuning sur ~10K exemples
- Le gain reste modeste car LIAR est intrinsequement difficile (textes courts, ambiguite, connaissances factuelles externes necessaires)

**Limites** :
- Sans GPU, l'entrainement et l'extraction d'embeddings sont lents
- Le fine-tuning sur ~10K exemples est sous-optimal pour des modeles aussi grands
- L'extraction d'embeddings ajoute une etape de preprocessing

**Prochaine etape** : Evaluation hors-domaine (ISOT, WELFake) pour tester la generalisation.